# Automated identification of head impact events in female and male football using computer vision and performance analysis annotations: A proof-of-concept.
**Luke Canavan Dignam · ATU Galway PhD · Sports Exercise and Nutrition**
Last updated: 22/09/2026

### V9 Design
- TSM-ResNet50 backbone, feature-caching pipeline, and evaluation harness as V7/V8
- Three parallel cohorts evaluated on every condition:
  - `unified` — all 17 development matches pooled (M02-M10 + W01-W08)
  - `male` — M02-M10 only (9 matches)
  - `female` — W01-W08 only (8 matches)
- Cross-gender transfer: Male→Female and Female→Male
- ablation dispatcher (Groups A–D)
- ** Group E — Spatial Augmentation Ablations** applied during feature extraction:
  - `E0` CenterCrop 224 (V7/V8 baseline)
  - `E1` RandomResizedCrop scale=0.7–1.0
  - `E2` RRC + RandomHorizontalFlip p=0.5
  - `E3` RRC + ColorJitter (brightness/contrast/saturation/hue)
  - `E4` RRC + RandomGrayscale p=0.1
  - `E5` RRC + GaussianBlur σ=0.1–2.0
  - `E6` RRC + RandomAffine ±10°, translate 5%, scale 0.9–1.1
  - `E7` RRC + RandomPerspective distortion=0.2
  - `E8` RRC + RandomErasing p=0.3, 2–10%
  - `E9` RRC + HFlip + ColorJitter (composite)
- Each Group-E condition trains unified/male/female in parallel and saves a three-way comparison
- Multi-seed results persisted to rolling CSV after each condition
- W09 held out completely throughout

### Requirements Document: V9 Head Impact Event Identification Pipeline

**1. Environment & Hardware**
*   **Platform:** Google Colaboratory (or similar Jupyter/Python environment).
*   **Hardware:** A CUDA-enabled GPU with at least 15.5 GB of free VRAM is recommended (e.g., Tesla T4, standard on Colab). The code explicitly checks for CUDA and relies on `expandable_segments:True` for memory management.
*   **Storage:** Google Drive (mounted at `/content/drive`) is used as the primary storage backend for datasets, annotations, and derived outputs. Temporary local storage (`/tmp`) is used for feature caching.

**2. Core Programming Language**
*   **Python:** 3.8+ (implied by the environment and syntax).

**3. Required Python Packages**
The pipeline relies on several third-party libraries for deep learning, computer vision, data manipulation, and evaluation.

*   **Deep Learning:**
    *   `torch` (PyTorch) - Core framework for model building and training.
    *   `torchvision` - Used for pre-trained models (ResNet50) and image transformations.
    *   `torchmetrics` - For evaluation metrics.
*   **Computer Vision & Image Processing:**
    *   `opencv-python-headless` (`cv2`) - Used for video clip loading and frame extraction.
*   **Data Manipulation & Analysis:**
    *   `numpy` - Array operations.
    *   `pandas` - Manifest handling, result aggregation, and CSV operations.
    *   `scipy` - Specifically `scipy.stats.mannwhitneyu` for statistical significance testing.
*   **Evaluation & Metrics:**
    *   `scikit-learn` - For AP, AUC, F1 scores, confusion matrices, and label binarization.
*   **Utility & Visualization:**
    *   `matplotlib` - Plotting confusion matrices and comparison charts.
    *   `tqdm` - Progress bars.
    *   `openpyxl` - Required by pandas to read Excel annotation files.

**4. Installation Command**
To set up the environment, run:
```bash
pip install torch torchvision torchmetrics scikit-learn openpyxl opencv-python-headless tqdm pandas numpy scipy matplotlib
```

**5. Directory Structure Assumptions**
The script expects a specific folder structure on the mounted Google Drive:
*   **Base:** `/content/drive/MyDrive/Chapter 2/Chapter2_Data`
*   **Annotations:** `Base/Annotations/` (Contains `.xlsx` files named by match ID, e.g., `M02.xlsx`).
*   **Video Extraction:** `Base/Extraction/` (Contains subfolders `Male/` and `Female/` containing folders like `M02/`, `W01/` with `.mp4` clips).
*   **Outputs/Derived:** `Base/derived/` (For saving features, manifests, and results).

**6. Data Requirements**
*   **Input Videos:** Short MP4 clips of events.
*   **Annotations:** Excel files detailing the labels ('header', 'aerial duel', 'pce', etc.).
*   **Manifest:** A pre-existing V4 manifest CSV (`event_manifest_paired.csv`) is required to bootstrap the V9 pipeline.

## Cell 1 · Installs & Imports

In [ ]:
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

!pip install -q torchmetrics scikit-learn openpyxl opencv-python-headless tqdm

import gc, json, warnings, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.models as models
import torchvision.transforms as T
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.preprocessing import label_binarize
from scipy.stats import mannwhitneyu
from tqdm.notebook import tqdm
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.2 MB/s eta 0:00:00
Device: cuda
GPU: Tesla T4
VRAM free: 15.5 GB


## Cell 2 · Paths & Config
Edit only this cell if paths change.

In [ ]:
from pathlib import Path

BASE       = Path('/content/drive/MyDrive/Chapter 2/Chapter2_Data')
ANNOTDIR   = BASE / 'Annotations'
EXTRACTDIR = BASE / 'Extraction'
DERIVED    = BASE / 'derived'

V9FEATDIR  = DERIVED / 'features_v9'
OUTPUTDIR  = DERIVED / 'results_v9'
FEATCACHE  = Path('/tmp/v9_feat_cache')   # rebuilt each session

for d in [V9FEATDIR, OUTPUTDIR, FEATCACHE]:
    d.mkdir(parents=True, exist_ok=True)

V4MANIFEST = DERIVED / 'event_manifest_paired.csv'
V9MANIFEST = DERIVED / 'event_manifest_v9.csv'

# Match IDs — M01 excluded (corrupted); W09 is independent held-out test
MALE_MATCHES   = [f'M{i:02d}' for i in range(2, 11)]    # M02-M10
FEMALE_MATCHES = [f'W{i:02d}' for i in range(1, 9)]     # W01-W08
EXCLUDE_MATCHES = {'M01', 'W09'}

MATCHDIRS = {
    **{m: EXTRACTDIR / 'Male'   / m for m in MALE_MATCHES},
    **{f: EXTRACTDIR / 'Female' / f for f in FEMALE_MATCHES},
}

CLASSES   = ['header', 'aerial_duel', 'pce']
N_CLASSES = len(CLASSES)
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
LABEL_MAP = {
    'header': 'header', 'headers': 'header',
    'aerial duel': 'aerial_duel', 'aerial': 'aerial_duel',
    'aerial_duel': 'aerial_duel', 'aerialduel': 'aerial_duel',
    'pce': 'pce', 'possible concussion': 'pce', 'possibleconcussion': 'pce',
}
SKIP_LABELS = {
    'half 1 start', 'half 1 end', 'half 2 start', 'half 2 end',
    'match end', 'match start',
}

# Training hyper-parameters — identical to V7/V8
FPS                    = 30
CLIP_SEC               = 1.5
FEAT_DIM               = 2048
EPOCHS                 = 30
BATCH_SIZE             = 64
LR                     = 1e-3
WEIGHT_DECAY           = 1e-4
NUM_SEGMENTS           = 4
PATIENCE               = 10
SEEDS                  = [42, 7, 99]
SEEDS_SHORT            = [42]
N_AUG                  = 2
IMG_SIZE               = 224
TEMPORAL_JITTER_FRAMES = 3
SAFEGUARD_PCE_AP       = 0.10

print('Paths')
for name, p in [('BASE', BASE), ('ANNOTDIR', ANNOTDIR), ('EXTRACTDIR', EXTRACTDIR),
                ('V4MANIFEST', V4MANIFEST), ('OUTPUTDIR', OUTPUTDIR)]:
    status = '✓' if p.exists() else '✗'
    print(f'  {status} {name}: {p}')

Paths
  ✓ BASE: /content/drive/MyDrive/Chapter 2/Chapter2_Data
  ✓ ANNOTDIR: /content/drive/MyDrive/Chapter 2/Chapter2_Data/Annotations
  ✓ EXTRACTDIR: /content/drive/MyDrive/Chapter 2/Chapter2_Data/Extraction
  ✓ V4MANIFEST: /content/drive/MyDrive/Chapter 2/Chapter2_Data/derived/event_manifest_paired.csv
  ✓ OUTPUTDIR: /content/drive/MyDrive/Chapter 2/Chapter2_Data/derived/results_v9


## Cell 3 · Validate Match Folders

In [ ]:
all_ok = True
print('Match clip folders')
for match_id, match_dir in MATCHDIRS.items():
    gender  = 'male' if match_id.startswith('M') else 'female'
    exists  = match_dir.exists()
    n_clips = len(list(match_dir.glob('*.mp4'))) if exists else 0
    annot   = ANNOTDIR / f'{match_id}.xlsx'
    ok      = exists and annot.exists()
    print(f'  {"✓" if ok else "✗"} {gender:6s} {match_id}  clips={n_clips:3d}  '
          f'labels={"YES" if annot.exists() else "NO"}')
    if not ok: all_ok = False
print()
print('All match folders verified.' if all_ok else 'Some matches missing — check paths in Cell 2.')

Match clip folders
  ✓ male   M02  clips= 93  labels=YES
  ✓ male   M03  clips=108  labels=YES
  ✓ male   M04  clips= 92  labels=YES
  ✓ male   M05  clips= 64  labels=YES
  ✓ male   M06  clips= 57  labels=YES
  ✓ male   M07  clips= 60  labels=YES
  ✓ male   M08  clips=129  labels=YES
  ✓ male   M09  clips= 95  labels=YES
  ✓ male   M10  clips=116  labels=YES
  ✓ female W01  clips= 77  labels=YES
  ✓ female W02  clips= 76  labels=YES
  ✓ female W03  clips= 90  labels=YES
  ✓ female W04  clips= 62  labels=YES
  ✓ female W05  clips= 65  labels=YES
  ✓ female W06  clips= 91  labels=YES
  ✓ female W07  clips= 69  labels=YES
  ✓ female W08  clips= 71  labels=YES

All match folders verified.


## Cell 4 · Build V9 Manifest
Reuses `event_manifest_paired.csv` from V4. M01 and W09 excluded at build time.

In [ ]:
def build_v9_manifest(v4_path, n_aug=N_AUG):
    df = pd.read_csv(v4_path)
    print(f'V4 manifest loaded: {len(df)} rows')
    label_col    = 'label' if 'label' in df.columns else df.columns[-1]
    df[label_col] = (df[label_col].str.lower()
                     .str.replace(r'[-_]', ' ', regex=True).str.strip())
    df['label']   = df[label_col].map(LABEL_MAP)
    df            = df[df['label'].notna()].copy()
    df['class_idx'] = df['label'].map(CLASS2IDX).astype(int)
    df            = df[~df['match_id'].isin(EXCLUDE_MATCHES)].copy()
    if 'gender' not in df.columns:
        df['gender'] = df['match_id'].apply(
            lambda x: 'male' if x.startswith('M') else 'female')
    df['neg_type'] = 'positive'
    df['aug_copy_idx'] = 0
    pce_rows = df[df['label'] == 'pce'].copy()
    aug_rows = [pce_rows.assign(aug_copy_idx=i) for i in range(1, n_aug + 1)]
    v9 = pd.concat([df] + aug_rows, ignore_index=True)
    v9 = v9.sort_values(['match_id', 'event_n', 'aug_copy_idx']).reset_index(drop=True)
    v9.to_csv(V9MANIFEST, index=False)
    base = v9[v9['aug_copy_idx'] == 0]
    print(f'V9 manifest: {len(v9)} rows  ({len(df)} positives + {len(v9)-len(df)} PCE aug copies)')
    print('Class distribution (base):'); print(base['label'].value_counts().to_string())
    print('Gender split:');              print(base['gender'].value_counts().to_string())
    print(f'Unique matches: {v9["match_id"].nunique()}')
    return v9

v9df = build_v9_manifest(V4MANIFEST)

V4 manifest loaded: 1472 rows
V9 manifest: 1532 rows  (1470 positives + 62 PCE aug copies)
Class distribution (base):
label
header         1005
aerial_duel     434
pce              31
Gender split:
gender
male      813
female    657
Unique matches: 18


## Cell 5 · TSM-ResNet50 Encoder
ImageNet weights loaded automatically via torchvision.

In [ ]:
class TemporalShift(nn.Module):
    def __init__(self, n_segment=NUM_SEGMENTS, shift_div=8):
        super().__init__()
        self.n_segment = n_segment
        self.shift_div = shift_div

    def forward(self, x):
        nt, c, h, w = x.shape
        t = self.n_segment
        n = nt // t
        x    = x.view(n, t, c, h, w)
        fold = c // self.shift_div
        out  = x.clone()
        out[:, 1:,   :fold]     = x[:, :-1, :fold]
        out[:, 0,    :fold]     = 0
        out[:, :-1, fold:2*fold] = x[:, 1:, fold:2*fold]
        out[:, -1,  fold:2*fold] = 0
        return out.view(nt, c, h, w)


def make_tsm_resnet50(n_classes=N_CLASSES, n_segment=NUM_SEGMENTS, pretrained=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    base    = models.resnet50(weights=weights)
    for layer in [base.layer1, base.layer2, base.layer3, base.layer4]:
        layer[0] = nn.Sequential(TemporalShift(n_segment), layer[0])
    feat_dim = base.fc.in_features
    base.fc  = nn.Sequential(
        nn.Dropout(0.3), nn.Linear(feat_dim, 512),
        nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, n_classes))
    return base

m = make_tsm_resnet50().to(DEVICE)
x = torch.randn(NUM_SEGMENTS, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
assert m(x).shape == (NUM_SEGMENTS, N_CLASSES)
del m, x; gc.collect(); torch.cuda.empty_cache()
print(f'TSM-ResNet50 OK — output ({NUM_SEGMENTS}, {N_CLASSES})')

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


TSM-ResNet50 OK — output (4, 3)


## Cell 6 · Spatial Augmentation Transform Registry (Group E)

| Key | Transform | Rationale |
|-----|-----------|-----------|
| `E0_baseline` | CenterCrop 224 | V7/V8 default |
| `E1_rrc` | RandomResizedCrop | Zoom/framing invariance |
| `E2_rrc_hflip` | RRC + HFlip | Mirror-legal for football |
| `E3_rrc_colorjitter` | RRC + ColorJitter | Broadcast colour grading |
| `E4_rrc_grayscale` | RRC + Grayscale | Colour-invariant features |
| `E5_rrc_gaussblur` | RRC + GaussianBlur | Motion blur / low-res robustness |
| `E6_rrc_affine` | RRC + RandomAffine | Camera angle tolerance |
| `E7_rrc_perspective` | RRC + RandomPerspective | Camera position variation |
| `E8_rrc_erasing` | RRC + RandomErasing | Occlusion robustness |
| `E9_composite` | RRC + HFlip + ColorJitter | Best-of-breed combination |

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def _val():
    return T.Compose([T.ToPILImage(), T.Resize(256), T.CenterCrop(IMG_SIZE),
                      T.ToTensor(), T.Normalize(MEAN, STD)])

def _aug(*extra):
    return T.Compose([T.ToPILImage(), *extra, T.ToTensor(), T.Normalize(MEAN, STD)])

RRC = T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.75, 1.33))

AUG_TRANSFORMS = {
    'E0_baseline':        _val(),
    'E1_rrc':             _aug(RRC),
    'E2_rrc_hflip':       _aug(RRC, T.RandomHorizontalFlip(0.5)),
    'E3_rrc_colorjitter': _aug(RRC, T.ColorJitter(0.3, 0.3, 0.2, 0.05)),
    'E4_rrc_grayscale':   _aug(RRC, T.RandomGrayscale(0.1)),
    'E5_rrc_gaussblur':   _aug(RRC, T.GaussianBlur(5, (0.1, 2.0))),
    'E6_rrc_affine':      _aug(RRC, T.RandomAffine(10, (0.05, 0.05), (0.9, 1.1))),
    'E7_rrc_perspective': _aug(RRC, T.RandomPerspective(0.2, 0.5)),
    'E8_rrc_erasing':     T.Compose([T.ToPILImage(), RRC, T.ToTensor(),
                                     T.Normalize(MEAN, STD),
                                     T.RandomErasing(0.3, (0.02, 0.1))]),
    'E9_composite':       _aug(RRC, T.RandomHorizontalFlip(0.5),
                               T.ColorJitter(0.3, 0.3, 0.2, 0.05)),
}

print('Spatial augmentation registry:')
for k in AUG_TRANSFORMS:
    print(f'  {k}')

Spatial augmentation registry:
  E0_baseline
  E1_rrc
  E2_rrc_hflip
  E3_rrc_colorjitter
  E4_rrc_grayscale
  E5_rrc_gaussblur
  E6_rrc_affine
  E7_rrc_perspective
  E8_rrc_erasing
  E9_composite


## Cell 7 · Feature Pre-Extraction

Extracts TSM-ResNet50 (identity head) features for all clips. Run **once per session**.
Other Group-E conditions are extracted lazily in Cell 13.

In [ ]:
def sample_frames(clip_path, n_segment=NUM_SEGMENTS,
                  temporal_jitter=False, jitter_frames=TEMPORAL_JITTER_FRAMES):
    cap   = cv2.VideoCapture(str(clip_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release(); return None
    if temporal_jitter and total > 2 * jitter_frames:
        start = random.randint(0, jitter_frames)
        end   = total - random.randint(0, jitter_frames)
    else:
        start, end = 0, total
    indices = np.linspace(start, end - 1, n_segment, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret
                      else np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    cap.release()
    return frames


class ExtractionDataset(Dataset):
    def __init__(self, df, clip_transform):
        self.rows      = df.reset_index(drop=True)
        self.transform = clip_transform
    def __len__(self):  return len(self.rows)
    def __getitem__(self, idx):
        row    = self.rows.iloc[idx]
        frames = sample_frames(row['clip_path'], temporal_jitter=False)
        if frames is None:
            frames = [np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)] * NUM_SEGMENTS
        tensor = torch.stack([self.transform(f) for f in frames])
        return tensor, int(row['class_idx']), row['match_id']


@torch.no_grad()
def extract_and_cache(manifest_df, clip_transform, cache_dir, batch_size=16):
    base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for layer in [base.layer1, base.layer2, base.layer3, base.layer4]:
        layer[0] = nn.Sequential(TemporalShift(NUM_SEGMENTS), layer[0])
    base.fc = nn.Identity()
    base    = base.to(DEVICE).eval()

    ds = ExtractionDataset(manifest_df, clip_transform)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=2, pin_memory=True)

    all_feats, all_labels, all_matches = [], [], []
    for clips, labels, match_ids in tqdm(dl, desc='Extracting features'):
        B, T, C, H, W = clips.shape
        clips = clips.view(B * T, C, H, W).to(DEVICE)
        with torch.cuda.amp.autocast():
            feats = base(clips)
        feats = feats.view(B, T, -1).mean(dim=1).cpu().float()
        all_feats.append(feats)
        all_labels.extend(labels.tolist())
        all_matches.extend(match_ids)

    feats_tensor = torch.cat(all_feats)
    cache_dir.mkdir(parents=True, exist_ok=True)
    torch.save({'feats': feats_tensor,
                'labels': torch.tensor(all_labels),
                'matches': all_matches},
               cache_dir / 'features.pt')
    print(f'  Cached {len(all_labels)} clips → {cache_dir / "features.pt"}')
    del base; gc.collect(); torch.cuda.empty_cache()
    return feats_tensor, all_labels, all_matches


# ── Extract baseline (E0) — identical to V7/V8 ───────────────────────────
print('Extracting E0_baseline features (V7/V8 parity) …')
E0_CACHE = FEATCACHE / 'E0_baseline'
FEAT_TENSOR, FEAT_LABELS, FEAT_MATCHES = extract_and_cache(
    v9df, AUG_TRANSFORMS['E0_baseline'], E0_CACHE)
print(f'Feature tensor: {FEAT_TENSOR.shape}')
print('Extraction done. GPU free for fast head training.')

Extracting E0_baseline features (V7/V8 parity) …


Extracting features:   0%|          | 0/96 [00:00<?, ?it/s]

  Cached 1532 clips → /tmp/v9_feat_cache/E0_baseline/features.pt
Feature tensor: torch.Size([1532, 2048])
Extraction done. GPU free for fast head training.


## Cell 8 · Loss Functions & Feature Classifier Training Loop

- `ConditionalFocalLoss` γ=1 header/aerial, γ=2 PCE — V7/V8 default
- `WeightedFocalLoss` per-class inverse-frequency — V8 Group B1
- `LabelSmoothingLoss` ε=0.1 — V8 Group B2
- MLP head: LayerNorm → Linear(2048→256) → ReLU → Dropout(0.3) → Linear(256→3)

In [ ]:
class ConditionalFocalLoss(nn.Module):
    def __init__(self, gamma_common=1.0, gamma_pce=2.0, pce_idx=2):
        super().__init__()
        self.gamma_common = gamma_common; self.gamma_pce = gamma_pce; self.pce_idx = pce_idx
    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, reduction='none')
        pt    = torch.exp(-ce)
        gamma = torch.where(targets == self.pce_idx,
                            torch.full_like(ce, self.gamma_pce),
                            torch.full_like(ce, self.gamma_common))
        return ((1 - pt) ** gamma * ce).mean()


class WeightedFocalLoss(nn.Module):
    def __init__(self, class_counts, gamma=2.0):
        super().__init__()
        counts = torch.tensor(class_counts, dtype=torch.float32)
        w = 1.0 / (counts + 1e-6)
        self.weights = (w / w.sum() * len(counts)).to(DEVICE)
        self.gamma   = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__(); self.smoothing = smoothing
    def forward(self, logits, targets):
        return F.cross_entropy(logits, targets, label_smoothing=self.smoothing)


def compute_mAP(y_true, logits):
    probs = torch.softmax(torch.tensor(np.array(logits), dtype=torch.float32), dim=1).numpy()
    y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
    aps   = []
    for c in range(N_CLASSES):
        aps.append(average_precision_score(y_bin[:, c], probs[:, c])
                   if y_bin[:, c].sum() > 0 else 0.0)
    return float(np.mean(aps)), aps


class FeatDataset(Dataset):
    def __init__(self, feats, labels):
        self.feats  = feats  if isinstance(feats,  torch.Tensor) else torch.tensor(np.array(feats),  dtype=torch.float32)
        self.labels = labels if isinstance(labels, torch.Tensor) else torch.tensor(np.array(labels), dtype=torch.long)
    def __len__(self):        return len(self.labels)
    def __getitem__(self, i): return self.feats[i], self.labels[i]


def train_fold_fast(train_idx, val_idx, feats, labels, seed=42,
                    criterion=None, use_weighted_sampler=False):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    train_idx = torch.tensor(np.array(train_idx), dtype=torch.long)
    val_idx   = torch.tensor(np.array(val_idx),   dtype=torch.long)
    if not isinstance(feats,  torch.Tensor): feats  = torch.tensor(np.array(feats),  dtype=torch.float32)
    if not isinstance(labels, torch.Tensor): labels = torch.tensor(np.array(labels), dtype=torch.long)
    train_labels = labels[train_idx]

    if use_weighted_sampler:
        cc  = [(train_labels == c).sum().item() for c in range(N_CLASSES)]
        sw  = torch.tensor([1.0 / (cc[l.item()] + 1e-6) for l in train_labels])
        dl  = DataLoader(FeatDataset(feats[train_idx], train_labels),
                         BATCH_SIZE, sampler=WeightedRandomSampler(sw, len(sw), True), num_workers=0)
    else:
        dl  = DataLoader(FeatDataset(feats[train_idx], train_labels),
                         BATCH_SIZE, shuffle=True, num_workers=0)
    val_dl = DataLoader(FeatDataset(feats[val_idx], labels[val_idx]),
                        BATCH_SIZE, shuffle=False, num_workers=0)

    head = nn.Sequential(
        nn.LayerNorm(FEAT_DIM), nn.Linear(FEAT_DIM, 256),
        nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, N_CLASSES),
    ).to(DEVICE)
    if criterion is None: criterion = ConditionalFocalLoss()
    opt   = optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_loss, best_state, best_logits, patience_cnt = 1e9, None, None, 0
    for epoch in range(EPOCHS):
        head.train()
        for bx, by in dl:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad(); criterion(head(bx), by).backward(); opt.step()
        sched.step()
        head.eval()
        lv, yv = [], []
        with torch.no_grad():
            for vx, vy in val_dl:
                lv.append(head(vx.to(DEVICE)).cpu()); yv.extend(vy.tolist())
        lv = torch.cat(lv); yv = torch.tensor(yv)
        vl = F.cross_entropy(lv, yv).item()
        if vl < best_loss:
            best_loss, best_state, best_logits, patience_cnt = vl, {k: v.cpu().clone() for k, v in head.state_dict().items()}, lv.clone(), 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE: break

    mAP, per_class = compute_mAP(yv.tolist(), best_logits.numpy())
    return mAP, per_class, best_loss, yv.tolist(), best_logits.numpy().tolist()

print('ConditionalFocalLoss | WeightedFocalLoss | LabelSmoothingLoss')
print('FeatDataset + train_fold_fast ready.')

ConditionalFocalLoss | WeightedFocalLoss | LabelSmoothingLoss
FeatDataset + train_fold_fast ready.


## Cell 9 · LOOMO-CV & Transfer Runners

Only PCE-containing matches are eligible hold-out folds.

In [ ]:
def get_pce_matches(df):
    return set(df[df['label'] == 'pce']['match_id'].unique())


def run_loomo(manifest_df, feat_tensor, feat_labels, feat_matches,
              condition='unified', seeds=SEEDS_SHORT, verbose=True,
              criterion=None, use_weighted_sampler=False):
    base        = manifest_df[manifest_df['aug_copy_idx'] == 0].copy().reset_index(drop=True)
    pce_matches = get_pce_matches(base)
    match_arr   = np.array(feat_matches)
    results     = []
    if not hasattr(run_loomo, 'fold_results'):
        run_loomo.fold_results = []
    for seed in seeds:
        for test_match in tqdm(sorted(pce_matches), desc=f'{condition} s={seed}', leave=False):
            val_idx   = np.where(match_arr == test_match)[0]
            train_idx = np.where(match_arr != test_match)[0]
            if len(val_idx) == 0 or len(train_idx) < 5: continue
            mAP, per_class, val_loss, y_true, logits = train_fold_fast(
                train_idx, val_idx, feat_tensor,
                torch.tensor(feat_labels, dtype=torch.long),
                seed=seed, criterion=criterion,
                use_weighted_sampler=use_weighted_sampler)
            results.append({'condition': condition, 'test_match': test_match,
                            'mAP': mAP,
                            'AP_header': per_class[0] if per_class else np.nan,
                            'AP_aerial': per_class[1] if len(per_class) > 1 else np.nan,
                            'AP_pce':    per_class[2] if len(per_class) > 2 else np.nan,
                            'val_loss': val_loss, 'seed': seed})
            run_loomo.fold_results.append({'y_true': y_true, 'logits': np.array(logits),
                                           'test_match': test_match,
                                           'condition': condition, 'seed': seed})
            if verbose:
                print(f'  {test_match}  mAP={mAP:.3f}  '
                      f'header={per_class[0]:.3f}  aerial={per_class[1]:.3f}  '
                      f'pce={per_class[2]:.3f}')
    return pd.DataFrame(results)


def run_transfer(src_feats, src_labels, tgt_feats, tgt_labels, direction,
                 seeds=SEEDS_SHORT, criterion=None, use_weighted_sampler=False):
    combined_feats  = torch.cat([src_feats, tgt_feats])
    combined_labels = torch.cat([torch.tensor(src_labels, dtype=torch.long),
                                 torch.tensor(tgt_labels, dtype=torch.long)])
    train_idx = np.arange(len(src_labels))
    val_idx   = np.arange(len(src_labels), len(src_labels) + len(tgt_labels))
    results   = []
    for seed in seeds:
        mAP, per_class, val_loss, _, _ = train_fold_fast(
            train_idx, val_idx, combined_feats, combined_labels, seed=seed,
            criterion=criterion, use_weighted_sampler=use_weighted_sampler)
        results.append({'condition': direction, 'mAP': mAP,
                        'AP_header': per_class[0], 'AP_aerial': per_class[1],
                        'AP_pce':    per_class[2], 'val_loss': val_loss, 'seed': seed})
    return pd.DataFrame(results)

print('LOOMO-CV and Transfer runners ready.')

LOOMO-CV and Transfer runners ready.


## Cell 10 · Cohort Feature Split Helper

Splits any `(feat_tensor, feat_labels, feat_matches)` triple into male / female / unified subsets.

In [ ]:
def split_cohorts(feat_tensor, feat_labels, feat_matches, manifest_df):
    """Return (unified, male, female) triples of (feats, labels, matches, sub_df)."""
    match_arr    = np.array(feat_matches)
    male_set     = set(MALE_MATCHES)
    female_set   = set(FEMALE_MATCHES)

    male_mask    = np.array([m in male_set   for m in feat_matches])
    female_mask  = np.array([m in female_set for m in feat_matches])

    male_idx     = np.where(male_mask)[0]
    female_idx   = np.where(female_mask)[0]

    male_feats   = feat_tensor[male_idx]
    male_labels  = [feat_labels[i]  for i in male_idx]
    male_matches = [feat_matches[i] for i in male_idx]

    female_feats   = feat_tensor[female_idx]
    female_labels  = [feat_labels[i]  for i in female_idx]
    female_matches = [feat_matches[i] for i in female_idx]

    v9_male   = manifest_df[manifest_df['match_id'].isin(male_set)].copy()
    v9_female = manifest_df[manifest_df['match_id'].isin(female_set)].copy()

    unified = (feat_tensor, feat_labels, feat_matches, manifest_df)
    male    = (male_feats,   male_labels,   male_matches,   v9_male)
    female  = (female_feats, female_labels, female_matches, v9_female)

    print(f'  Unified clips: {len(feat_labels)}  '
          f'Male: {len(male_labels)}  Female: {len(female_labels)}')
    return unified, male, female

# Split baseline features
E0_UNIFIED, E0_MALE, E0_FEMALE = split_cohorts(
    FEAT_TENSOR, FEAT_LABELS, FEAT_MATCHES, v9df)
print('Cohort split ready.')

  Unified clips: 1532  Male: 843  Female: 624
Cohort split ready.


## Cell 11 · V6/V8 Ablation Conditions (Groups A–D)

Carried forward from V8 unchanged — used as the baseline loss/sampler conditions.

In [ ]:
base_labels_arr = np.array([FEAT_LABELS[i] for i in range(len(FEAT_LABELS))
                             if v9df.iloc[i]['aug_copy_idx'] == 0])
CLASS_COUNTS = np.array([(base_labels_arr == c).sum() for c in range(N_CLASSES)])
print(f'Class counts (base): {dict(zip(CLASSES, CLASS_COUNTS))}')

ABLATION_CONDITIONS = {
    'A0_baseline': {
        'criterion': ConditionalFocalLoss(),
        'use_weighted_sampler': False,
        'description': 'V7 baseline — ConditionalFocalLoss, no sampler',
    },
    'B1_weighted_focal': {
        'criterion': WeightedFocalLoss(CLASS_COUNTS),
        'use_weighted_sampler': False,
        'description': 'WeightedFocalLoss inverse-frequency, γ=2',
    },
    'B2_label_smooth': {
        'criterion': LabelSmoothingLoss(0.1),
        'use_weighted_sampler': False,
        'description': 'LabelSmoothing ε=0.1',
    },
    'C1_weighted_sampler': {
        'criterion': ConditionalFocalLoss(),
        'use_weighted_sampler': True,
        'description': 'WeightedRandomSampler + ConditionalFocalLoss',
    },
    'C2_focal_plus_sampler': {
        'criterion': WeightedFocalLoss(CLASS_COUNTS),
        'use_weighted_sampler': True,
        'description': 'WeightedFocalLoss + WeightedRandomSampler',
    },
}
for k, v in ABLATION_CONDITIONS.items():
    print(f'  {k:30s}  {v["description"]}')
print('\nAblation dispatcher (Groups A-D) ready.')

Class counts (base): {'header': np.int64(1005), 'aerial_duel': np.int64(434), 'pce': np.int64(31)}
  A0_baseline                     V7 baseline — ConditionalFocalLoss, no sampler
  B1_weighted_focal               WeightedFocalLoss inverse-frequency, γ=2
  B2_label_smooth                 LabelSmoothing ε=0.1
  C1_weighted_sampler             WeightedRandomSampler + ConditionalFocalLoss
  C2_focal_plus_sampler           WeightedFocalLoss + WeightedRandomSampler

Ablation dispatcher (Groups A-D) ready.


## Cell 12 · Three-Way Runner (Unified / Male / Female)

`run_three_way()` runs LOOMO-CV on all three cohorts **in a single call**, appends results to
the rolling CSV, and returns a tidy summary DataFrame.  Every ablation condition — Groups A–D
and each Group-E spatial aug — uses this function so results are directly comparable.

In [ ]:
OUT_CSV = OUTPUTDIR / 'v9_results.csv'

def _save(df):
    if OUT_CSV.exists():
        pd.concat([pd.read_csv(OUT_CSV), df], ignore_index=True).to_csv(OUT_CSV, index=False)
    else:
        df.to_csv(OUT_CSV, index=False)


def run_three_way(condition_tag, unified, male, female,
                  seeds=SEEDS_SHORT, criterion=None, use_weighted_sampler=False,
                  verbose=False):
    """Run LOOMO-CV on unified / male / female cohorts.
    Each cohort is identified by (feats, labels, matches, manifest_df).
    Returns a summary DataFrame with one row per cohort."""
    all_dfs = []
    for cohort_name, (ft, fl, fm, mdf) in [
            ('unified', unified), ('male', male), ('female', female)]:

        cond_label = f'{condition_tag}_{cohort_name}'
        print(f'\n{"="*55}')
        print(f'  {cond_label}')
        print(f'{"="*55}')

        df_res = run_loomo(mdf, ft, fl, fm,
                           condition=cond_label, seeds=seeds, verbose=verbose,
                           criterion=criterion,
                           use_weighted_sampler=use_weighted_sampler)

        mean_pce = df_res['AP_pce'].mean() if len(df_res) > 0 else 0.0
        df_res['pce_ap_flag'] = mean_pce < SAFEGUARD_PCE_AP
        df_res['aug_tag']     = condition_tag
        df_res['cohort']      = cohort_name
        _save(df_res)
        all_dfs.append(df_res)

        g = df_res[['mAP', 'AP_header', 'AP_aerial', 'AP_pce']]
        flag = ' ⚠ safeguard' if mean_pce < SAFEGUARD_PCE_AP else ''
        print(f'  mAP={g["mAP"].mean():.3f}±{g["mAP"].std():.3f}  '
              f'Header={g["AP_header"].mean():.3f}  '
              f'Aerial={g["AP_aerial"].mean():.3f}  '
              f'PCE={g["AP_pce"].mean():.3f}{flag}')

    combined = pd.concat(all_dfs, ignore_index=True)

    # Print three-way summary table
    print(f'\n  {"Cohort":<10}  {"mAP":>6}  {"±":>6}  {"Header":>8}  {"Aerial":>8}  {"PCE":>6}')
    print('  ' + '-' * 52)
    for cohort in ['unified', 'male', 'female']:
        sub = combined[combined['cohort'] == cohort]
        if len(sub) == 0: continue
        print(f'  {cohort:<10}  {sub["mAP"].mean():>6.3f}  {sub["mAP"].std():>6.3f}  '
              f'{sub["AP_header"].mean():>8.3f}  {sub["AP_aerial"].mean():>8.3f}  '
              f'{sub["AP_pce"].mean():>6.3f}')
    return combined


def run_condition_three_way(condition_name,
                             unified=None, male=None, female=None,
                             seeds=SEEDS_SHORT, verbose=False):
    """Dispatch a named Groups A-D condition through the three-way runner."""
    if unified is None: unified = E0_UNIFIED
    if male    is None: male    = E0_MALE
    if female  is None: female  = E0_FEMALE
    cfg = ABLATION_CONDITIONS[condition_name]
    return run_three_way(condition_name, unified, male, female,
                          seeds=seeds, criterion=cfg['criterion'],
                          use_weighted_sampler=cfg['use_weighted_sampler'],
                          verbose=verbose)

print('run_three_way + run_condition_three_way ready.')

run_three_way + run_condition_three_way ready.


## Cell 13 · Run Groups A–D (Three-Way)

Runs every V8 ablation condition on unified / male / female in sequence.

In [ ]:
# Run all Groups A-D
ab_results = {}
for cname in ABLATION_CONDITIONS:
    print(f'\n>>> {cname}: {ABLATION_CONDITIONS[cname]["description"]}')
    ab_results[cname] = run_condition_three_way(cname)
print('\nAll Groups A-D complete.')


>>> A0_baseline: V7 baseline — ConditionalFocalLoss, no sampler

  A0_baseline_unified


A0_baseline_unified s=42:   0%|          | 0/15 [00:00<?, ?it/s]

  mAP=0.367±0.041  Header=0.680  Aerial=0.298  PCE=0.123

  A0_baseline_male


A0_baseline_male s=42:   0%|          | 0/8 [00:00<?, ?it/s]

  mAP=0.346±0.018  Header=0.683  Aerial=0.290  PCE=0.064 ⚠ safeguard

  A0_baseline_female


...

  mAP=0.359±0.026  Header=0.687  Aerial=0.281  PCE=0.108

  C2_focal_plus_sampler_male


C2_focal_plus_sampler_male s=42:   0%|          | 0/8 [00:00<?, ?it/s]

  mAP=0.333±0.005  Header=0.675  Aerial=0.261  PCE=0.064 ⚠ safeguard

  C2_focal_plus_sampler_female


C2_focal_plus_sampler_female s=42:   0%|          | 0/6 [00:00<?, ?it/s]

  mAP=0.375±0.028  Header=0.680  Aerial=0.298  PCE=0.147

  Cohort         mAP       ±    Header    Aerial     PCE
  ----------------------------------------------------
  unified      0.359   0.026     0.687     0.281   0.108
  male         0.333   0.005     0.675     0.261   0.064
  female       0.375   0.028     0.680     0.298   0.147

All Groups A-D complete.


## Cell 14 · Run Group E — Spatial Augmentation Ablations (Three-Way)

For each aug condition: (1) extract features → (2) split cohorts → (3) run three-way LOOMO-CV.

In [ ]:
AUG_RESULTS = {}     # aug_key → combined 3-way DataFrame

def load_or_extract(aug_key, force_reextract=False):
    cache_dir = FEATCACHE / aug_key
    feat_file = cache_dir / 'features.pt'
    if feat_file.exists() and not force_reextract:
        print(f'  Loading cached features for {aug_key} …')
        saved = torch.load(feat_file)
        return saved['feats'], saved['labels'].tolist(), saved['matches']
    print(f'  Extracting features for {aug_key} …')
    return extract_and_cache(v9df, AUG_TRANSFORMS[aug_key], cache_dir)


# E0_baseline already extracted — register it directly
print('Registering E0_baseline (already extracted) …')
AUG_RESULTS['E0_baseline'] = run_three_way(
    'E0_baseline', E0_UNIFIED, E0_MALE, E0_FEMALE,
    seeds=SEEDS_SHORT, criterion=ConditionalFocalLoss(),
    use_weighted_sampler=False, verbose=False)

# E1–E9: extract then run
for aug_key in list(AUG_TRANSFORMS.keys())[1:]:
    print(f'\n>>> {aug_key}')
    ft, fl, fm = load_or_extract(aug_key)
    print(f'  Feature tensor: {ft.shape}')
    unified_e, male_e, female_e = split_cohorts(ft, fl, fm, v9df)
    AUG_RESULTS[aug_key] = run_three_way(
        aug_key, unified_e, male_e, female_e,
        seeds=SEEDS_SHORT, criterion=ConditionalFocalLoss(),
        use_weighted_sampler=False, verbose=False)
    del ft; gc.collect(); torch.cuda.empty_cache()

print('\nAll Group E conditions complete.')

Registering E0_baseline (already extracted) …

  E0_baseline_unified


E0_baseline_unified s=42:   0%|          | 0/15 [00:00<?, ?it/s]

  mAP=0.367±0.041  Header=0.680  Aerial=0.298  PCE=0.123

  E0_baseline_male


E0_baseline_male s=42:   0%|          | 0/8 [00:00<?, ?it/s]

  mAP=0.346±0.018  Header=0.683  Aerial=0.290  PCE=0.064 ⚠ safeguard

  E0_baseline_female


E0_baseline_female s=42:   0%|          | 0/6 [00:00<?, ?it/s]

  mAP=0.385±0.027  Header=0.675  Aerial=0.328  PCE=0.154

  Cohort         mAP       ±    Header    Aerial     PCE
  ----------------------------------------------------
  unified      0.367   0.041     0.680     0.298   0.123
  male         0.346   0.018     0.683     0.290   0.064
  female       0.385   0.027     0.675     0.328   0.154

>>> E1_rrc
  Extracting features for E1_rrc …


Extracting features:   0%|          | 0/96 [00:00<?, ?it/s]

## Cell 15 · Cross-Gender Transfer (Male→Female, Female→Male)

Uses the best Group-E aug condition (by unified PCE AP). Falls back to E0_baseline.

In [ ]:
# Find best Group-E condition by unified PCE AP
df_all = pd.read_csv(OUT_CSV)
aug_unified = df_all[(df_all['cohort'] == 'unified') &
                     (df_all['aug_tag'].str.startswith('E'))]

if len(aug_unified) > 0:
    best_aug = aug_unified.groupby('aug_tag')['AP_pce'].mean().idxmax()
    print(f'Best Group-E condition (unified PCE AP): {best_aug}')
else:
    best_aug = 'E0_baseline'
    print('No Group-E results yet — using E0_baseline.')

# Load cached features for best aug
best_cache = FEATCACHE / best_aug / 'features.pt'
if best_cache.exists() and best_aug != 'E0_baseline':
    saved = torch.load(best_cache)
    best_ft, best_fl, best_fm = saved['feats'], saved['labels'].tolist(), saved['matches']
else:
    best_ft, best_fl, best_fm = FEAT_TENSOR, FEAT_LABELS, FEAT_MATCHES

_, (m_ft, m_fl, m_fm, _), (f_ft, f_fl, f_fm, _) = split_cohorts(
    best_ft, best_fl, best_fm, v9df)

print(f'\nMale clips: {len(m_fl)}   Female clips: {len(f_fl)}')

print('\nMale → Female transfer …')
mf_res = run_transfer(m_ft, m_fl, f_ft, f_fl, 'male_to_female', seeds=SEEDS_SHORT)
mf_res['aug_tag'] = best_aug

print('Female → Male transfer …')
fm_res = run_transfer(f_ft, f_fl, m_ft, m_fl, 'female_to_male', seeds=SEEDS_SHORT)
fm_res['aug_tag'] = best_aug

transfer_res = pd.concat([mf_res, fm_res], ignore_index=True)
transfer_res.to_csv(OUTPUTDIR / 'v9_transfer_results.csv', index=False)

for _, row in transfer_res.iterrows():
    print(f'  {row["condition"]:20s}  mAP={row["mAP"]:.3f}  '
          f'PCE={row["AP_pce"]:.3f}  seed={row["seed"]}')

# Mann-Whitney gender gap test
df_all2 = pd.read_csv(OUT_CSV)
m_pce = df_all2[(df_all2['cohort'] == 'male')   & (df_all2['aug_tag'] == 'E0_baseline')]['AP_pce'].dropna().values
f_pce = df_all2[(df_all2['cohort'] == 'female') & (df_all2['aug_tag'] == 'E0_baseline')]['AP_pce'].dropna().values
if len(m_pce) >= 2 and len(f_pce) >= 2:
    stat, p = mannwhitneyu(m_pce, f_pce, alternative='two-sided')
    r = 1 - 2 * stat / (len(m_pce) * len(f_pce))
    print(f'\nMann-Whitney U (PCE AP, male vs female): U={stat:.1f}  p={p:.4f}  r={r:.3f}')
    print('Significant gender gap detected.' if p < 0.05 else 'No significant gender gap.')

## Cell 16 · W09 Independent Held-Out Test

Train on all 17 development matches, test on W09. Uses best aug condition. W09 has never been seen during any training or CV fold.

In [ ]:
W09_DIR      = EXTRACTDIR / 'Female' / 'W09'
W09_CACHE    = FEATCACHE / 'w09'
W09_CACHE.mkdir(exist_ok=True)

# Build W09 manifest
recs = []
for p in sorted(W09_DIR.glob('*.mp4')):
    stem  = p.stem.lower().replace('_', ' ').replace('-', ' ')
    label = next((m for raw, m in LABEL_MAP.items()
                  if raw.replace('_', ' ') in stem), None)
    if label:
        recs.append({'clip_path': str(p), 'class_idx': CLASS2IDX[label],
                     'label': label, 'match_id': 'W09', 'aug_copy_idx': 0})
w09df_test = pd.DataFrame(recs)
print(f'W09: {len(w09df_test)} clips'); print(w09df_test['label'].value_counts().to_string())

# Extract using best aug transform
best_transform = AUG_TRANSFORMS.get(best_aug, AUG_TRANSFORMS['E0_baseline'])
w09_ft, w09_fl_raw, _ = extract_and_cache(w09df_test, best_transform, W09_CACHE, 16)

# Train on all 17 dev matches, test on W09
n_tr = len(best_fl); n_te = len(w09_fl_raw)
comb_feats  = torch.cat([best_ft, w09_ft])
comb_labels = torch.cat([torch.tensor(best_fl, dtype=torch.long),
                          torch.tensor(w09_fl_raw, dtype=torch.long)])
train_idx = np.arange(n_tr); test_idx = np.arange(n_tr, n_tr + n_te)

w09_res = []
for seed in SEEDS:
    mAP, pc, vl, yt, lg = train_fold_fast(train_idx, test_idx, comb_feats, comb_labels, seed)
    w09_res.append({'seed': seed, 'mAP': mAP,
                    'AP_header': pc[0], 'AP_aerial': pc[1], 'AP_pce': pc[2],
                    'y_true': yt, 'logits': np.array(lg)})
    print(f'  seed={seed}  mAP={mAP:.3f}  header={pc[0]:.3f}  aerial={pc[1]:.3f}  pce={pc[2]:.3f}')

# Ensemble
all_y    = np.array(w09_res[0]['y_true'])
all_logits = np.mean([r['logits'] for r in w09_res], axis=0)
all_probs  = np.exp(all_logits) / np.exp(all_logits).sum(1, keepdims=True)
all_preds  = all_probs.argmax(1)
y_bin      = label_binarize(all_y, classes=list(range(N_CLASSES)))

print(f'\nW09 seed-avg mAP: {np.mean([r["mAP"] for r in w09_res]):.4f} '
      f'± {np.std([r["mAP"] for r in w09_res]):.4f}')
for i, cls in enumerate(CLASSES):
    ap  = average_precision_score(y_bin[:, i], all_probs[:, i])
    auc = roc_auc_score(y_bin[:, i], all_probs[:, i]) if y_bin[:, i].sum() > 0 else float('nan')
    f1  = f1_score(all_y, all_preds, labels=[i], average='macro', zero_division=0)
    print(f'  {cls:<15}  AP={ap:.4f}  AUC={auc:.4f}  F1={f1:.4f}')

cm      = confusion_matrix(all_y, all_preds, labels=list(range(N_CLASSES)))
cm_norm = cm.astype(float) / cm.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mat, title, fmt in zip(axes, [cm, cm_norm],
                                ['W09 Raw Counts', 'W09 Row-Normalised (Recall)'], ['d', '.2f']):
    ConfusionMatrixDisplay(mat, display_labels=CLASSES).plot(ax=ax, colorbar=False, cmap='Blues', values_format=fmt)
    ax.set_title(title, fontsize=11, fontweight='bold'); ax.tick_params(axis='x', rotation=20)
plt.suptitle('W09 Independent Held-Out Test', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUTDIR / 'v9_w09_confusion.png', dpi=150, bbox_inches='tight'); plt.show()

pd.DataFrame([{k: v for k, v in r.items() if k not in ['y_true', 'logits']}
               for r in w09_res]).to_csv(OUTPUTDIR / 'v9_w09_results.csv', index=False)
print(f'Saved → {OUTPUTDIR / "v9_w09_results.csv"}')

## Cell 17 · Three-Way Comparison Plot (Unified vs Male vs Female)

Mirrors V7 Cell 13 — grouped bar chart per condition, one panel per metric.

In [ ]:
df_all = pd.read_csv(OUT_CSV)
metrics = ['AP_header', 'AP_aerial', 'AP_pce']
titles  = ['Header AP', 'Aerial Duel AP', 'PCE AP']
colors  = {'unified': '#01696f', 'male': '#006494', 'female': '#a12c7b'}

# Unique condition tags (aug or ablation)
tags = df_all['aug_tag'].unique() if 'aug_tag' in df_all.columns else df_all['condition'].str.rsplit('_', n=1).str[0].unique()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('V9 Three-Way Comparison — Unified / Male / Female', fontsize=13, fontweight='bold')

for ax, metric, title in zip(axes, metrics, titles):
    grouped = df_all.groupby(['aug_tag', 'cohort'])[metric].mean().reset_index()
    tag_list = sorted(grouped['aug_tag'].unique())
    x = np.arange(len(tag_list))
    w = 0.25
    for j, cohort in enumerate(['unified', 'male', 'female']):
        sub = grouped[grouped['cohort'] == cohort]
        vals = [sub[sub['aug_tag'] == t][metric].values[0]
                if t in sub['aug_tag'].values else 0.0 for t in tag_list]
        ax.bar(x + j * w, vals, w, label=cohort, color=colors[cohort], alpha=0.85)
    ax.set_xticks(x + w)
    ax.set_xticklabels(tag_list, rotation=40, ha='right', fontsize=7)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Average Precision')
    ax.set_title(title, fontweight='bold')
    ax.axhline(SAFEGUARD_PCE_AP, color='red', linestyle=':', linewidth=1.2,
               label=f'Safeguard ({SAFEGUARD_PCE_AP})')
    ax.legend(fontsize=7)

plt.tight_layout()
save_path = OUTPUTDIR / 'v9_three_way_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

## Cell 18 · Group E Ranking — PCE AP by Cohort

Ranks all spatial aug conditions by unified PCE AP, overlaying male and female scores.

In [ ]:
df_all = pd.read_csv(OUT_CSV)
e_rows = df_all[df_all['aug_tag'].str.startswith('E')]

if len(e_rows) > 0:
    pivot = e_rows.groupby(['aug_tag', 'cohort'])['AP_pce'].mean().unstack('cohort')
    pivot = pivot.sort_values('unified', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(pivot))
    w = 0.25
    for j, cohort in enumerate(['unified', 'male', 'female']):
        if cohort in pivot.columns:
            ax.bar(x + j * w, pivot[cohort], w, label=cohort,
                   color=colors[cohort], alpha=0.85)
    ax.set_xticks(x + w)
    ax.set_xticklabels(pivot.index, rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('PCE Average Precision')
    ax.set_title('Group E — Spatial Augmentation PCE AP Ranking (highest unified first)',
                 fontweight='bold')
    ax.axhline(SAFEGUARD_PCE_AP, color='red', linestyle=':', linewidth=1.2,
               label=f'Safeguard ({SAFEGUARD_PCE_AP})')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUTDIR / 'v9_group_e_pce_ranking.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Best aug condition (unified PCE):')
    print(pivot['unified'].sort_values(ascending=False).head(3).to_string())
else:
    print('No Group E results yet — run Cell 14 first.')

## Cell 19 · Statistical Significance (Mann-Whitney U)

Pairwise test: each Group-E aug condition vs E0_baseline, per cohort, for PCE AP.

In [ ]:
df_all = pd.read_csv(OUT_CSV)
baseline_tag = 'E0_baseline'

for cohort in ['unified', 'male', 'female']:
    base_pce = df_all[(df_all['aug_tag'] == baseline_tag) &
                      (df_all['cohort']  == cohort)]['AP_pce'].dropna().values
    print(f'\n{cohort.upper()} — Mann-Whitney vs {baseline_tag}  (n={len(base_pce)}, '
          f'mean={base_pce.mean():.3f})')
    print(f'  {"Condition":<35}  {"Mean PCE":>9}  {"U":>8}  {"p":>8}  sig?')
    print('  ' + '-' * 66)
    for tag in sorted(df_all['aug_tag'].unique()):
        if tag == baseline_tag: continue
        pce = df_all[(df_all['aug_tag'] == tag) &
                     (df_all['cohort']  == cohort)]['AP_pce'].dropna().values
        if len(pce) < 2 or len(base_pce) < 2: continue
        stat, p = mannwhitneyu(pce, base_pce, alternative='two-sided')
        sig = '✓' if p < 0.05 else ''
        print(f'  {tag:<35}  {pce.mean():>9.3f}  {stat:>8.1f}  {p:>8.4f}  {sig}')

## Cell 20 · Save Final Model Weights

Train on all 17 development matches (best aug, V7 defaults) and save weights for deployment.

In [ ]:
WEIGHTS_DIR = DERIVED / 'weights_v9'
WEIGHTS_DIR.mkdir(exist_ok=True)

for seed in SEEDS:
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    head = nn.Sequential(
        nn.LayerNorm(FEAT_DIM), nn.Linear(FEAT_DIM, 256),
        nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, N_CLASSES),
    ).to(DEVICE)
    ds  = FeatDataset(best_ft, torch.tensor(best_fl, dtype=torch.long))
    dl  = DataLoader(ds, BATCH_SIZE, shuffle=True, num_workers=0)
    opt = optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = ConditionalFocalLoss()
    for epoch in range(EPOCHS):
        head.train()
        for bx, by in dl:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad(); crit(head(bx), by).backward(); opt.step()
        sch.step()
    save_path = WEIGHTS_DIR / f'head_seed{seed}.pt'
    torch.save(head.state_dict(), save_path)
    print(f'Saved {save_path}')

config = {'N_CLASSES': N_CLASSES, 'FEAT_DIM': FEAT_DIM, 'CLASSES': CLASSES,
          'CLASS2IDX': CLASS2IDX, 'NUM_SEGMENTS': NUM_SEGMENTS,
          'IMG_SIZE': IMG_SIZE, 'SEEDS': SEEDS, 'best_aug': best_aug}
with open(WEIGHTS_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved → {WEIGHTS_DIR / "config.json"}')

## Cell 21 · GradCAM Interpretability

Visualises which spatial regions drive predictions. Fill in a real clip path to run.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.grads = None; self.acts = None
        target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'acts', o.detach()))
        target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0].detach()))

    def __call__(self, x, class_idx):
        self.model.eval()
        logits = self.model(x)
        self.model.zero_grad()
        logits[0, class_idx].backward()
        w   = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(dim=1, keepdim=True))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.squeeze().cpu().numpy()


def visualise_gradcam(clip_path, model, class_idx, class_name, save_path=None):
    frames = sample_frames(clip_path, temporal_jitter=False)
    if not frames:
        print(f'Could not load {clip_path}'); return
    transform = AUG_TRANSFORMS['E0_baseline']
    tensors   = torch.stack([transform(f) for f in frames]).to(DEVICE)
    gc_tool   = GradCAM(model, model.layer4[-1])
    cam       = gc_tool(tensors[:1], class_idx)
    cam_rgb   = cv2.resize(cam, (frames[0].shape[1], frames[0].shape[0]))
    overlay   = cv2.applyColorMap((cam_rgb * 255).astype(np.uint8), cv2.COLORMAP_JET)
    overlay   = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    blended   = (0.55 * np.array(frames[0], dtype=np.float32) +
                 0.45 * overlay.astype(np.float32)).astype(np.uint8)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(frames[0]); axes[0].set_title('Input frame'); axes[0].axis('off')
    axes[1].imshow(blended);   axes[1].set_title(f'GradCAM — {class_name}'); axes[1].axis('off')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150)
    plt.show()

# ── Example (fill in a real clip path): ───────────────────────────────────
# full_model = make_tsm_resnet50().to(DEVICE)
# visualise_gradcam(
#     clip_path='/content/drive/MyDrive/Chapter 2/Chapter2Data/Extraction/Male/M04/M04MaleAerialDuel0044t3403.0s.mp4',
#     model=full_model, class_idx=CLASS2IDX['aerial_duel'], class_name='aerial_duel',
#     save_path=OUTPUTDIR / 'gradcam_aerial.png')
print('GradCAM ready — uncomment example above with a real clip path.')